In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
)

In [ ]:
os.getcwd()

In [ ]:
df = pd.read_csv(r'/right_arm.csv')

df.head()

In [ ]:
#replace blank spaces in the column names with '_'
df.columns = df.columns.str.replace(' ', '_')

In [ ]:
import ast

# Detect which columns contain list-type values
def is_list_column(series):
    sample = series.dropna().iloc[0]
    if isinstance(sample, list):
        return True
    if isinstance(sample, str):
        try:
            return isinstance(ast.literal_eval(sample), list)
        except:
            return False
    return False

list_columns = [col for col in df.columns if is_list_column(df[col])]
print(f"Found {len(list_columns)} list-format columns:")
for col in list_columns:
    print(f"  - {col}")

In [ ]:
expanded_dfs = [df.drop(columns=list_columns)]  # start with non-list columns

for col in list_columns:
    # Parse string to list if needed
    parsed = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    
    # Get number of elements in the list
    n_elements = len(parsed.iloc[0])
    
    # Create clean column names from original column name
    clean_name = col.strip().replace(' ', '_')
    new_col_names = [f"{clean_name}_{i+1}" for i in range(n_elements)]
    
    # Expand into separate columns
    expanded = pd.DataFrame(parsed.tolist(), columns=new_col_names, index=df.index)
    expanded_dfs.append(expanded)
    
    print(f"✔ '{col}' → expanded into {n_elements} columns: {new_col_names}")

# Combine everything into one flat dataframe
df_expanded = pd.concat(expanded_dfs, axis=1)

print(f"\nOriginal shape : {df.shape}")
print(f"Expanded shape : {df_expanded.shape}")
df_expanded.head()

In [ ]:
# Columns to always exclude from features
exclude_cols = ['Anomaly_State']

# Original scalar columns you want to keep
scalar_features = ['Norm_of_Cartesion_Linear_Momentum', 'Robot_Current',
                   'Tool_Current', 'Tool_Temperature', 'TCP_Force', 'Execution_Time']

# Auto-collect all expanded list column names
expanded_features = [col for col in df_expanded.columns
                     if any(col.startswith(c.strip().replace(' ', '_'))
                            for c in list_columns)]

# Combine all features
all_features = scalar_features + expanded_features

X = df_expanded[all_features]
y = df_expanded['Anomaly_State']

print(f"Total features : {len(all_features)}")
print(f"Feature list   :")
for f in all_features:
    print(f"  - {f}")

In [ ]:
print("X shape :", X.shape)
print("y shape :", y.shape)
print("\nAny nulls in X:", X.isnull().sum().sum())
print("Any nulls in y:", y.isnull().sum())
print("\nSample of expanded features:")
X.head(3)

In [ ]:
# ── 2. Train / Test Split (80 / 20) ──────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y          # preserves class balance in both splits
)
 
print(f"Training samples : {len(X_train)}")
print(f"Testing  samples : {len(X_test)}\n")
 

In [ ]:
# ── 3. Model Definition & Training ───────────────────────────────────────────
rf_model = RandomForestClassifier(
    n_estimators=100,       # number of trees
    max_depth=None,         # grow trees until pure leaves
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1               # use all available CPU cores
)
 
rf_model.fit(X_train, y_train)
print("Model training complete.\n")

In [ ]:
# ── 4. Predictions ────────────────────────────────────────────────────────────
y_pred = rf_model.predict(X_test)
 
# ── 5. Evaluation Metrics ─────────────────────────────────────────────────────
accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall    = recall_score(y_test, y_pred,    average='weighted', zero_division=0)
f1        = f1_score(y_test, y_pred,        average='weighted', zero_division=0)
cm        = confusion_matrix(y_test, y_pred)
 
print("=" * 45)
print("         EVALUATION METRICS")
print("=" * 45)
print(f"  Accuracy  : {accuracy:.4f}  ({accuracy * 100:.2f}%)")
print(f"  Precision : {precision:.4f}")
print(f"  Recall    : {recall:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print("=" * 45)
 
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))
 
print("Confusion Matrix:")
print(cm)

In [ ]:
# ── 6. Confusion Matrix Plot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=rf_model.classes_)
disp.plot(cmap='Blues', ax=ax, colorbar=True)
ax.set_title(f"Confusion Matrix (Accuracy: {accuracy * 100:.2f}%)", fontsize=14, fontweight='normal')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()
print("\nConfusion matrix saved as 'confusion_matrix.png'.")

In [ ]:
# ── 7. Feature Importance Plot ────────────────────────────────────────────────
feature_names = X.columns.tolist()
importances   = rf_model.feature_importances_
indices       = np.argsort(importances)[::-1]
 
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(range(len(feature_names)),
       importances[indices],
       color='steelblue', edgecolor='white')
ax.set_xticks(range(len(feature_names)))
ax.set_xticklabels([feature_names[i] for i in indices], rotation=90, ha='right')
ax.set_ylabel('Importance Score')
ax.set_title('Feature Importances — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importances.png', dpi=250)
plt.show()
print("Feature importance plot saved as 'feature_importances.png'.")

In [ ]:
# ── 8. Per-feature importance table ──────────────────────────────────────────
importance_df = (
    pd.DataFrame({'Feature': feature_names,
                  'Importance': importances})
    .sort_values('Importance', ascending=False)
    .reset_index(drop=True)
)
print("\nFeature Importance Ranking:")
print(importance_df.to_string(index=False))

In [ ]:
import shap
print(f"SHAP version: {shap.__version__}")

In [ ]:
# Retrain the model with fewer trees — 50 is usually enough for SHAP
rf_model = RandomForestClassifier(
    n_estimators=50,       # ← reduced from 100
    max_depth=10,          # ← limit tree depth (big speed improvement)
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

explainer = shap.TreeExplainer(rf_model)

In [ ]:
# Compute SHAP values on the test set
# For large datasets, use a sample to keep it fast
sample_size = min(1000, len(X_test))   # use up to 1000 rows
X_sample = X_test.sample(n=sample_size, random_state=42)
 
print(f"Computing SHAP values on {sample_size} samples...")
shap_values = explainer.shap_values(X_sample)
 
# For binary classification, shap_values is a list [class_0, class_1]
# We take class_1 (anomaly = 1) for interpretation
if isinstance(shap_values, list):
    shap_vals = shap_values[1]   # class 1 = Anomaly
    expected_value = explainer.expected_value[1]
else:
    shap_vals = shap_values
    expected_value = explainer.expected_value
 
print(f"SHAP values shape: {shap_vals.shape}")
print("SHAP computation complete.")

In [ ]:
# ── Step 1: Auto-handle all SHAP output formats ───────────────────────────────
print(f"Raw SHAP type  : {type(shap_values)}")

if isinstance(shap_values, list):
    # Format A: list of arrays → one per class [(n, f), (n, f)]
    shap_vals_plot = shap_values[1]
    base_val       = explainer.expected_value[1]
    print(f"Format: list → selected class 1")

elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    # Format B: 3D array → (samples, features, classes)
    shap_vals_plot = shap_values[:, :, 1]
    base_val       = explainer.expected_value[1]
    print(f"Format: 3D array → sliced [:, :, 1]")

elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 2:
    # Format C: already 2D (regression or single output)
    shap_vals_plot = shap_values
    base_val       = explainer.expected_value
    print(f"Format: 2D array → used directly")

else:
    raise ValueError(f"Unexpected SHAP values format: {type(shap_values)}, shape: {np.array(shap_values).shape}")

print(f"Final SHAP shape : {shap_vals_plot.shape}")   # must be (n_samples, n_features)
print(f"Base value       : {base_val}")

# ── Step 2: Build Explanation object ─────────────────────────────────────────
shap_explanation = shap.Explanation(
    values=shap_vals_plot,
    base_values=base_val,
    data=X_sample.values,
    feature_names=X_sample.columns.tolist()
)

# ── Step 3: Beeswarm Plot ─────────────────────────────────────────────────────
plt.figure(figsize=(10, 7))
shap.plots.beeswarm(shap_explanation, max_display=12, show=False)
plt.title('SHAP Beeswarm Plot — Anomaly Prediction',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_beeswarm.png")

In [ ]:
# Clean summary of which features matter most overall
print("\nGenerating SHAP Bar Plot...")
 
plt.figure(figsize=(9, 6))
shap.plots.bar(
    shap_explanation,
    max_display=12,
    show=False
)
plt.title('SHAP Mean Absolute Impact per Feature',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: shap_bar.png")

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
import matplotlib.pyplot as plt
import numpy as np

# ── Top features by SHAP importance ──────────────────────────────────────────
mean_shap     = np.abs(shap_vals_plot).mean(axis=0)
feature_names = X_sample.columns.tolist()
top_n         = min(12, len(feature_names))
top_indices   = np.argsort(mean_shap)[::-1][:top_n].tolist()   # .tolist() is critical
top_features  = [feature_names[i] for i in top_indices]

print(f"Plotting PDPs for: {top_features}")

# ── PDP Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 10))

PartialDependenceDisplay.from_estimator(
    rf_model,
    X_sample,
    features=top_indices,               # use integer indices
    feature_names=feature_names,
    kind='average',                     # 'average' = PDP only
    grid_resolution=50,
    ax=axes.flatten()[:top_n],
    random_state=42
)

# Hide unused subplots
for i in range(top_n, len(axes.flatten())):
    axes.flatten()[i].set_visible(False)

fig.suptitle('Partial Dependence Plots — Top Features by SHAP Importance',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('pdp_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: pdp_plots.png")

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# ── Top features by SHAP importance ──────────────────────────────────────────
mean_shap     = np.abs(shap_vals_plot).mean(axis=0)
feature_names = X_sample.columns.tolist()
top_n         = min(12, len(feature_names))
top_indices   = np.argsort(mean_shap)[::-1][:top_n].tolist()
top_features  = [feature_names[i] for i in top_indices]

print(f"Plotting PDPs for: {top_features}")

# ── PDP Plot ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 10))
flat_axes = axes.flatten()

disp = PartialDependenceDisplay.from_estimator(
    rf_model,
    X_sample,
    features=top_indices,
    feature_names=feature_names,
    kind='average',
    grid_resolution=50,
    ax=flat_axes[:top_n],
    random_state=42
)

# ── Add red reference point on each subplot ───────────────────────────────────
for i, (feat_idx, feat_name) in enumerate(zip(top_indices, top_features)):
    ax = flat_axes[i]

    # Reference x = mean of the feature
    ref_x = float(X_sample[feat_name].mean())

    # Reference y = model prediction at mean feature value (baseline probability)
    ref_y = rf_model.predict_proba(X_sample)[:, 1].mean()

    # Vertical dashed line at mean feature value
    ax.axvline(
        x=ref_x,
        color='red',
        linestyle='--',
        linewidth=1.5,
        alpha=0.7,
        label='Mean value'
    )

    # Horizontal dashed line at baseline prediction
    ax.axhline(
        y=ref_y,
        color='orange',
        linestyle=':',
        linewidth=1.5,
        alpha=0.7,
        label='Baseline prediction'
    )

    # Red dot at the intersection of mean x and baseline y
    ax.scatter(
        ref_x, ref_y,
        color='red',
        s=100,                  # dot size
        zorder=5,               # draw on top of everything
        label=f'Reference\n({ref_x:.2f}, {ref_y:.3f})'
    )

    # Annotate the red dot with its coordinates
    ax.annotate(
        f'  ({ref_x:.2f}, {ref_y:.3f})',
        xy=(ref_x, ref_y),
        fontsize=8,
        color='red',
        va='bottom'
    )

    ax.legend(fontsize=8, loc='best')
    # ax.set_title(f'PDP — {feat_name}', fontsize=10, fontweight='bold')

# Hide unused subplots
for i in range(top_n, len(flat_axes)):
    flat_axes[i].set_visible(False)

fig.suptitle('Partial Dependence Plots — Red dot = mean feature value × baseline prediction',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('pdp_with_reference.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: pdp_with_reference.png")

In [ ]:
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=(16, 10))

PartialDependenceDisplay.from_estimator(
    rf_model,
    X_sample,
    features=top_indices,
    feature_names=feature_names,
    kind='both',                        # 'both' = PDP + ICE curves
    subsample=200,                      # limit ICE lines for speed
    grid_resolution=50,
    ax=axes.flatten()[:top_n],
    random_state=42,
    ice_lines_kw={'alpha': 0.05,        # ICE lines very transparent
                  'color': 'steelblue'},
    pd_line_kw={'color': 'red',         # mean PDP line in red
                'linewidth': 2.5}
)

for i in range(top_n, len(axes.flatten())):
    axes.flatten()[i].set_visible(False)

fig.suptitle('PDP + ICE Plots — Top Features by SHAP Importance',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('pdp_ice_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: pdp_ice_plots.png")

In [ ]:
# ── SHAP Feature Importance Summary Table ─────────────────────────────────────

# Use shap_vals_plot (2D array) not shap_vals (raw 3D array)
mean_shap_abs = np.abs(shap_vals_plot).mean(axis=0)   # mean absolute SHAP
mean_shap_raw = shap_vals_plot.mean(axis=0)            # mean signed SHAP

# Verify shapes match feature names before building DataFrame
assert len(feature_names) == len(mean_shap_abs), \
    f"Mismatch: {len(feature_names)} features vs {len(mean_shap_abs)} SHAP values"

shap_summary = (
    pd.DataFrame({
        'Rank'          : range(1, len(feature_names) + 1),
        'Feature'       : feature_names,
        'Mean_Abs_SHAP' : mean_shap_abs,
        'Mean_SHAP'     : mean_shap_raw,
        'Direction'     : ['↑ Increases anomaly' if v > 0
                           else '↓ Decreases anomaly'
                           for v in mean_shap_raw]
    })
    .sort_values('Mean_Abs_SHAP', ascending=False)
    .reset_index(drop=True)
)

shap_summary.index = range(1, len(shap_summary) + 1)   # rank from 1

print("\nSHAP Feature Importance Ranking:")
print(shap_summary.to_string())